# 01 – Cleaning van de mushroom-dataset

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk *(vul teamleden aan)* |
| **Wat** | De ruwe projectdataset inlezen, onderzoeken en omzetten naar één propere dataset. Op het einde leggen we ook de vaste train/test-split vast die alle modellen in `02_modeling.ipynb` (en later AWS) gebruiken. |
| **Input** | `data/raw/mushroom_project_dataset.csv` |
| **Output** | `data/clean/mushrooms_clean.parquet`, `data/clean/mushrooms_clean.csv`, `data/clean/test_ids.json` |

## Over de dataset
De docent heeft een **aangepaste versie** van de [Secondary Mushroom dataset (UCI)](https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset) gemaakt. Volgens de opdracht is die:
- ingekort: **5.000 rijen** (in plaats van 61.069) en **minder kenmerken**;
- aangevuld met **ontbrekende waarden**;
- mogelijk voorzien van **omgedraaide labels** (eetbaar ↔ giftig).

We gebruiken de UCI-versie zelf **niet**. De UCI-pagina gebruiken we enkel als documentatie (betekenis van de lettercodes en eenheden).

Elke stap volgt hetzelfde patroon:
1. **Vaststelling** – wat zien we in de data (met code als bewijs)?
2. **Keuze** – wat doen we ermee?
3. **Waarom** – de motivatie, en waarom we het alternatief *niet* kozen.

> ⚠️ Deze notebook moet uitgevoerd worden met `mushrooms/` als werkmap (standaard in Jupyter/VS Code als je de notebook daar opent). Ze bevat bewust geen grafieken: die horen in de EDA.

## 0. Imports en paden

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split

RAW_PATH = Path("data/raw/mushroom_project_dataset.csv")
OUT_DIR = Path("data/clean")
SEED = 42

pd.set_option("display.max_columns", 50)

## 1. Data inlezen
**Vaststelling:** in tegenstelling tot de UCI-versie is dit bestand gescheiden met `,` (niet `;`), dus `read_csv` werkt zonder extra parameters.

In [2]:
raw = pd.read_csv(RAW_PATH)
print(raw.shape)
raw.head()

(5000, 13)


,cap-diameter,stem-height,stem-width,spore-print-color,gill-color,habitat,season,ring-type,cap-shape,stem-surface,jumbled_noise_0,jumbled_noise_1,class
0,NaN,6.42,13.51,NaN,NaN,d,u,g,x,NaN,x,c,p
1,6.99,7.68,18.76,NaN,w,d,a,f,f,s,f,f,e
2,5.78,7.67,8.21,NaN,r,d,u,f,c,NaN,c,x,p
3,6.34,8.34,11.06,NaN,NaN,d,a,z,x,NaN,x,b,p
4,NaN,5.67,12.78,NaN,n,d,NaN,f,x,NaN,s,s,e


In [3]:
raw.dtypes

cap-diameter         float64
stem-height          float64
stem-width           float64
spore-print-color     object
gill-color            object
habitat               object
season                object
ring-type             object
cap-shape             object
stem-surface          object
jumbled_noise_0       object
jumbled_noise_1       object
class                 object
dtype: object

**Vaststelling:** 13 kolommen: 3 numerieke afmetingen, 7 categorische kenmerken, de target `class` en twee verdachte kolommen `jumbled_noise_0` en `jumbled_noise_1`. Elke categorische waarde is een lettercode (bv. `x` = convex).

## 2. Kolomnamen

**Vaststelling:**
- Koppeltekens (`cap-diameter`) zijn onhandig in Python: `df.cap-diameter` werkt niet.
- De eenheden staan enkel op de UCI-pagina: `cap-diameter` en `stem-height` zijn in **cm**, maar `stem-width` in **mm**.

**Keuze:** alles naar lowercase `snake_case`, en de eenheid toevoegen aan de drie numerieke kolommen.

**Waarom:** eenheden in de naam voorkomen fouten in het invulformulier van de frontend (een gebruiker die de steelbreedte in cm ingeeft, zou een 10× te kleine waarde doorsturen).

In [4]:
UNIT_RENAME = {
    "cap_diameter": "cap_diameter_cm",
    "stem_height": "stem_height_cm",
    "stem_width": "stem_width_mm",
}

df = raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace("-", "_")
df = df.rename(columns=UNIT_RENAME)
df.columns.tolist()

['cap_diameter_cm',
 'stem_height_cm',
 'stem_width_mm',
 'spore_print_color',
 'gill_color',
 'habitat',
 'season',
 'ring_type',
 'cap_shape',
 'stem_surface',
 'jumbled_noise_0',
 'jumbled_noise_1',
 'class']

## 3. De kolommen `jumbled_noise_0` en `jumbled_noise_1`

**Vaststelling 1:** de naam zegt het al, maar we willen het bewijzen. Beide kolommen hebben **exact dezelfde waarden en aantallen** als `cap_shape`, inclusief evenveel ontbrekende waarden:

In [5]:
NOISE_COLS = ["jumbled_noise_0", "jumbled_noise_1"]
pd.DataFrame({c: df[c].value_counts(dropna=False) for c in ["cap_shape"] + NOISE_COLS})

,cap_shape,jumbled_noise_0,jumbled_noise_1
x,2060,2060,2060
f,1078,1078,1078
s,533,533,533
NaN,406,406,406
b,354,354,354
o,218,218,218
p,207,207,207
c,144,144,144


**Vaststelling 2:** toch staan de waarden op **andere rijen**. Als de kolommen willekeurig door elkaar geschudde kopieën van `cap_shape` zijn, dan komen ze per rij enkel overeen *bij toeval*.
Die toevalskans kunnen we berekenen: de kans dat twee willekeurige rijen dezelfde vorm hebben = de som van de kwadraten van de aandelen.

In [6]:
shares = df["cap_shape"].value_counts(normalize=True, dropna=False)
chance = (shares[shares.index.notna()] ** 2).sum()

print(f"Verwachte overeenkomst bij puur toeval: {chance:.1%}")
for col in NOISE_COLS:
    print(f"{col}: zelfde waarde als cap_shape op {(df[col] == df['cap_shape']).mean():.1%} van de rijen, "
          f"NaN op dezelfde rij: {(df[col].isna() & df['cap_shape'].isna()).sum()} van {df[col].isna().sum()}")

Verwachte overeenkomst bij puur toeval: 23.7%
jumbled_noise_0: zelfde waarde als cap_shape op 23.9% van de rijen, NaN op dezelfde rij: 26 van 406
jumbled_noise_1: zelfde waarde als cap_shape op 23.5% van de rijen, NaN op dezelfde rij: 42 van 406


**Vaststelling 3:** hangen ze samen met de klasse? We gebruiken de **chi-kwadraattoets** (hangt de verdeling af van de klasse?) en **Cramér's V** (hoe sterk is dat verband, van 0 = geen tot 1 = perfect).
We vergelijken met de echte categorische kolommen.

In [7]:
def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2, p, _, _ = chi2_contingency(table)
    return np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1))), p


CATEGORICAL_RAW = [c for c in df.select_dtypes("object").columns if c != "class"]
association = pd.DataFrame(
    [cramers_v(df[c].fillna("NaN"), df["class"]) for c in CATEGORICAL_RAW],
    index=CATEGORICAL_RAW, columns=["cramers_v", "p_value"],
).sort_values("cramers_v", ascending=False)
association.round(4)

,cramers_v,p_value
stem_surface,0.2318,0.0000
cap_shape,0.1651,0.0000
ring_type,0.1619,0.0000
spore_print_color,0.1348,0.0000
gill_color,0.1279,0.0000
habitat,0.1131,0.0000
season,0.0747,0.0000
jumbled_noise_1,0.0595,0.0133
jumbled_noise_0,0.0341,0.5609


**Interpretatie:**
- De overeenkomst met `cap_shape` (±24%) is exact wat je bij toeval verwacht, en de NaN's staan op andere rijen. De kolommen zijn dus **`cap_shape`, rij per rij door elkaar geschud**. De waarde op een rij zegt niets over die paddenstoel.
- Het verband met de klasse is het zwakst van alle kolommen (Cramér's V ≈ 0,03–0,06 tegenover 0,11–0,23 voor de echte kenmerken).
- `jumbled_noise_1` heeft toch een p-waarde onder 0,05. Dat is een mooi voorbeeld van **toeval bij veel testen**: we doen 9 toetsen, en een geschudde kolom kan per ongeluk een klein verband tonen in deze 5.000 rijen. Met een Bonferroni-correctie (0,05 / 9 ≈ 0,006) is het niet meer significant. Het effect is verwaarloosbaar en kan logisch niet echt zijn.

**Keuze:** beide kolommen verwijderen.

**Waarom:**
- Pure ruis geeft een model (zeker een beslissingsboom) extra kansen om toevallige patronen in de trainset te **overfitten**.
- Een gebruiker van de webapp kan deze "kenmerken" niet invullen: ze bestaan niet in het echt.

In [8]:
df = df.drop(columns=NOISE_COLS)
NUMERIC_COLS = list(UNIT_RENAME.values())
CATEGORICAL_COLS = [c for c in df.columns if c not in NUMERIC_COLS + ["class"]]
print("numeriek   :", NUMERIC_COLS)
print("categorisch:", CATEGORICAL_COLS)

numeriek   : ['cap_diameter_cm', 'stem_height_cm', 'stem_width_mm']
categorisch: ['spore_print_color', 'gill_color', 'habitat', 'season', 'ring_type', 'cap_shape', 'stem_surface']


## 4. De target `class`

**Vaststelling:** twee klassen, `e` (eetbaar) en `p` (giftig), zonder ontbrekende waarden. De verdeling is ongeveer **62% eetbaar / 38% giftig**.
In de UCI-versie was dat 45/55: de docent heeft dus niet willekeurig rijen gekozen, of de labels zijn deels omgedraaid (of allebei).

In [9]:
print(df["class"].value_counts(dropna=False))
df["class"].value_counts(normalize=True).round(3)

class
e    3107
p    1893
Name: count, dtype: int64


class
e    0.621
p    0.379
Name: proportion, dtype: float64

**Omgedraaide labels?** Die kunnen we in de cleaning **niet** betrouwbaar opsporen: er is geen soortkolom meer en geen enkele rij komt dubbel voor met een andere klasse (zie stap 5).
**Keuze:** labels laten zoals ze zijn.

**Gevolg voor de modellen:** een deel van de labels is waarschijnlijk fout, ook in de testset. Een score van 100% is dus onmogelijk (en zou verdacht zijn). Een model dat de trainset perfect uit het hoofd leert, leert ook de foute labels. Regularisatie (bv. `min_samples_leaf` bij bomen) wordt daarom belangrijk.

## 5. Duplicaten

**Vaststelling:** in het ruwe bestand zijn er geen duplicaten. Na het verwijderen van de ruiskolommen wel een paar:

In [10]:
print("Duplicaten in het ruwe bestand          :", raw.duplicated().sum())
print("Duplicaten na verwijderen ruiskolommen  :", df.duplicated().sum())
print("Zelfde kenmerken, maar andere klasse    :", df.duplicated(subset=NUMERIC_COLS + CATEGORICAL_COLS).sum() - df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(NUMERIC_COLS + CATEGORICAL_COLS, na_position="first")

Duplicaten in het ruwe bestand          : 0
Duplicaten na verwijderen ruiskolommen  : 5
Zelfde kenmerken, maar andere klasse    : 0


,cap_diameter_cm,stem_height_cm,stem_width_mm,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,class
316,NaN,NaN,NaN,NaN,NaN,d,a,f,f,NaN,e
3608,NaN,NaN,NaN,NaN,NaN,d,a,f,f,NaN,e
1770,NaN,NaN,NaN,NaN,w,d,NaN,f,x,NaN,e
3851,NaN,NaN,NaN,NaN,w,d,NaN,f,x,NaN,e
580,NaN,0.0,0.0,NaN,NaN,d,a,f,o,f,p
4082,NaN,0.0,0.0,NaN,NaN,d,a,f,o,f,p
232,NaN,0.0,0.0,NaN,f,d,w,f,o,f,p
3164,NaN,0.0,0.0,NaN,f,d,w,f,o,f,p
4244,NaN,0.0,0.0,NaN,f,d,w,f,o,f,p


**Interpretatie:** alle "duplicaten" hebben 2 tot 6 ontbrekende waarden, waaronder altijd `cap_diameter_cm` (de enige kolom met een uniek, continu getal). Het zijn dus hoogstwaarschijnlijk **verschillende paddenstoelen** die er door de gaten in de data identiek uitzien.
Er zijn geen conflicterende duplicaten (zelfde kenmerken, andere klasse).

**Keuze:** duplicaten **behouden**.

**Waarom:** een echte kopie zouden we verwijderen (geen nieuwe info, wel leakage tussen train en test). Maar dit zijn aparte waarnemingen, en met maar 5.000 rijen telt elke rij (*"make sure to use each row optimally"*).
In de webapp komt precies zo'n onvolledige invoer ook voor.

## 6. Ontbrekende waarden (NaN)

### 6a. Hoeveel ontbreekt er?

In [11]:
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "pct_missing": (missing / len(df) * 100).round(1)}).sort_values("pct_missing", ascending=False)
print(f"Rijen met minstens één NaN: {df.isna().any(axis=1).sum()} van {len(df)}")
missing_table

Rijen met minstens één NaN: 4988 van 5000


,n_missing,pct_missing
spore_print_color,4740,94.8
stem_surface,3359,67.2
cap_diameter_cm,2032,40.6
stem_height_cm,1013,20.3
habitat,1004,20.1
gill_color,989,19.8
season,973,19.5
ring_type,577,11.5
cap_shape,406,8.1
stem_width_mm,395,7.9


**Vaststelling:** alleen `class` is volledig. `spore_print_color` (95%) en `stem_surface` (67%) zijn bijna leeg, de andere kolommen missen 8–41%. Bijna **elke rij** heeft minstens één gat, dus rijen met NaN verwijderen is geen optie.

### 6b. Is "ontbrekend" zelf informatief?
Er zijn twee soorten gaten:
- **Structureel**: het kenmerk werd voor die soort nooit beschreven (in de UCI-versie waren `spore_print_color` en `stem_surface` al grotendeels leeg).
- **Willekeurig** (*missing completely at random*): de docent heeft waarden weggegooid.

Het verschil is belangrijk. Een willekeurig gat zegt niets over de paddenstoel, een structureel gat mogelijk wel. We toetsen per kolom of het % giftig verschilt tussen rijen *met* en *zonder* waarde:

In [12]:
y_poisonous = df["class"].eq("p")
rows = []
for col in NUMERIC_COLS + CATEGORICAL_COLS:
    is_missing = df[col].isna()
    rows.append({
        "kolom": col,
        "% giftig als ontbrekend": round(y_poisonous[is_missing].mean() * 100, 1),
        "% giftig als aanwezig": round(y_poisonous[~is_missing].mean() * 100, 1),
        "p_value": chi2_contingency(pd.crosstab(is_missing, y_poisonous))[1],
    })
pd.DataFrame(rows).set_index("kolom").sort_values("p_value").round(4)

,% giftig als ontbrekend,% giftig als aanwezig,p_value
kolom,,,
stem_surface,35.5,42.8,0.0000
spore_print_color,37.3,48.8,0.0002
habitat,40.2,37.3,0.0887
cap_shape,35.7,38.0,0.3807
ring_type,36.2,38.1,0.4140
cap_diameter_cm,38.3,37.5,0.5856
season,38.6,37.7,0.5999
stem_width_mm,37.0,37.9,0.7419
gill_color,38.1,37.8,0.8799


En ontbreken waarden samen (bv. altijd de hele steel tegelijk)?

In [13]:
nan_corr = df[NUMERIC_COLS + CATEGORICAL_COLS].isna().corr()
print(f"Grootste correlatie tussen twee NaN-patronen: {nan_corr.where(~np.eye(len(nan_corr), dtype=bool)).abs().max().max():.2f}")

Grootste correlatie tussen twee NaN-patronen: 0.08


**Interpretatie:**
- Enkel bij `stem_surface` en `spore_print_color` hangt "ontbrekend" samen met de klasse (p < 0,001). Dat zijn de **structurele** gaten.
- Bij alle andere kolommen is het % giftig gelijk, of de waarde nu ontbreekt of niet → **willekeurig** weggegooid.
- De NaN-patronen zijn onderling niet gecorreleerd (max ±0,08): elke kolom werd apart leeggemaakt.

### 6c. Logische reconstructie: paddenstoelen zonder steel

**Vaststelling:** sommige paddenstoelen hebben geen steel (bv. korstzwammen die direct op hout groeien). Dan zijn `stem_height_cm = 0`, `stem_width_mm = 0` én `stem_surface = f` (none).
We controleren eerst of die regel nergens geschonden wordt, en kijken dan hoeveel gaten we ermee kunnen invullen:

In [14]:
no_stem = (df["stem_height_cm"] == 0) | (df["stem_width_mm"] == 0) | (df["stem_surface"] == "f")

# Regel: zodra één kolom "geen steel" zegt, mogen de andere enkel ook "geen steel" zeggen of ontbreken
stem_values = df.loc[no_stem, ["stem_height_cm", "stem_width_mm", "stem_surface"]]
assert stem_values["stem_height_cm"].isin([0]).where(stem_values["stem_height_cm"].notna(), True).all()
assert stem_values["stem_width_mm"].isin([0]).where(stem_values["stem_width_mm"].notna(), True).all()
assert stem_values["stem_surface"].eq("f").where(stem_values["stem_surface"].notna(), True).all()

print(f"Rijen zonder steel: {no_stem.sum()} – geen enkele tegenspraak")
stem_values.isna().sum().rename("in te vullen")

Rijen zonder steel: 60 – geen enkele tegenspraak

stem_height_cm    11
stem_width_mm      3
stem_surface       4
Name: in te vullen, dtype: int64

**Keuze:** bij deze rijen de ontbrekende steelwaarden invullen met `0` / `f`.

**Waarom:** dit is geen gok maar een **logische afleiding**: als de steel 0 cm hoog is, dan is hij ook 0 mm breed en heeft hij geen oppervlak. Bij 60 rijen zonder steel is de regel nooit geschonden.

In [15]:
df.loc[no_stem, ["stem_height_cm", "stem_width_mm"]] = df.loc[no_stem, ["stem_height_cm", "stem_width_mm"]].fillna(0.0)
df.loc[no_stem, "stem_surface"] = df.loc[no_stem, "stem_surface"].fillna("f")
df.loc[no_stem, ["stem_height_cm", "stem_width_mm", "stem_surface"]].isna().sum().sum()

np.int64(0)

### 6d. Categorische kolommen: eigen categorie `"unknown"`

**Keuze:** ontbrekende categorische waarden worden de categorie `"unknown"`. We verwijderen geen kolommen.

**Waarom niet de alternatieven?**

| Alternatief | Waarom niet |
|---|---|
| Rijen met NaN verwijderen | 99,8% van de rijen heeft minstens één NaN. |
| Invullen met de modus (meest voorkomende waarde) | We verzinnen kenmerken. Bij `stem_surface`/`spore_print_color` zou 67–95% van de rijen dezelfde verzonnen waarde krijgen, en het (informatieve) signaal "niet beschreven" gaat verloren. |
| `spore_print_color` verwijderen (95% leeg) | De 5% ingevulde waarden hangen sterk samen met de klasse (Cramér's V 0,14, zie stap 3), én "ontbrekend" zelf is informatief (stap 6b). |

- Bij de **structurele** kolommen is `"unknown"` een betekenisvolle categorie.
- Bij de **willekeurige** kolommen draagt `"unknown"` geen signaal, maar het kan ook geen kwaad. Het model leert dat het dan op de andere kenmerken moet steunen.
- In de webapp kan de gebruiker zo gewoon *"weet ik niet"* kiezen.

**Belangrijk onderscheid:** `f` (= "none", bv. geen steel) is een **echte waarneming**, geen ontbrekende waarde. `"unknown"` = niet ingevuld, `none` = beschreven als afwezig. Die houden we bewust apart.

In [16]:
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].fillna("unknown")
df[CATEGORICAL_COLS].isna().sum().sum()

np.int64(0)

### 6e. Numerieke kolommen: NaN laten staan

**Keuze:** de ontbrekende afmetingen blijven `NaN` in de propere dataset. Ze worden pas **in de modelpipeline** ingevuld (mediaan, `SimpleImputer`).

**Waarom:**
- De mediaan moet berekend worden op de **trainset alleen**. Als we hier op de volledige dataset invullen, lekt informatie uit de testset naar de training.
- Omdat de gaten willekeurig zijn (6b), verwachten we geen voordeel van een extra "was ontbrekend"-kolom. Een eenvoudige mediaan volstaat.
- `cap_diameter_cm` mist 41%, maar de ingevulde waarden zijn nuttig. De kolom verwijderen zou die informatie weggooien.

In [17]:
df.isna().sum()[lambda s: s > 0]

cap_diameter_cm    2032
stem_height_cm     1002
stem_width_mm       392
dtype: int64

## 7. Uitschieters in de numerieke kolommen

In [18]:
df[NUMERIC_COLS].describe().round(2)

,cap_diameter_cm,stem_height_cm,stem_width_mm
count,2968.00,3998.00,4608.00
mean,7.16,6.68,12.88
std,5.99,3.47,10.61
min,0.50,0.00,0.00
25%,3.83,4.79,5.94
50%,6.14,6.00,11.27
75%,8.78,7.73,17.05
max,57.40,32.43,102.48


In [19]:
q1, q3 = df[NUMERIC_COLS].quantile(0.25), df[NUMERIC_COLS].quantile(0.75)
iqr_outlier = (df[NUMERIC_COLS] < q1 - 1.5 * (q3 - q1)) | (df[NUMERIC_COLS] > q3 + 1.5 * (q3 - q1))
print("Negatieve waarden        :", (df[NUMERIC_COLS] < 0).sum().to_dict())
print("Uitschieters volgens IQR :", iqr_outlier.sum().to_dict())
df.nlargest(5, "cap_diameter_cm")[NUMERIC_COLS + ["cap_shape", "habitat", "class"]]

Negatieve waarden        : {'cap_diameter_cm': 0, 'stem_height_cm': 0, 'stem_width_mm': 0}
Uitschieters volgens IQR : {'cap_diameter_cm': 134, 'stem_height_cm': 283, 'stem_width_mm': 164}


,cap_diameter_cm,stem_height_cm,stem_width_mm,cap_shape,habitat,class
3262,57.40,9.29,57.22,o,unknown,e
3519,55.49,7.69,56.96,unknown,d,e
3007,55.16,7.98,43.28,o,unknown,e
2181,54.98,NaN,44.03,o,unknown,e
280,54.81,NaN,25.02,o,d,e


**Interpretatie:**
- Geen negatieve waarden. Een steel van 0 is een paddenstoel zonder steel (stap 6c), geen meetfout.
- De IQR-regel markeert honderden rijen, maar dat komt door de **scheve verdeling** (veel kleine paddenstoelen, enkele grote soorten), niet door fouten.
- De grootste hoeden (±55 cm) zijn allemaal eetbaar, met hoedvorm `o` (others) en habitat `d` (woods) waar die ingevuld zijn. Dat is één consistente, echt grote soort (vermoedelijk de zadelzwam, *Dryad's Saddle*, die zo groot wordt), geen willekeurige fout.

**Keuze:** **geen** uitschieters verwijderen of afknippen (clipping).

**Waarom:** het zijn echte (gesimuleerde) paddenstoelen. De IQR-regel zou juist de grootste soorten wegfilteren, en daar is het model dan blind voor. Bomen hebben geen last van grote waarden.
Transformaties die sommige modellen wel nodig hebben (schalen voor logistische regressie of KNN) gebeuren **in de modelpipeline**.

## 8. Codes vertalen naar leesbare labels

**Vaststelling:** alle categorieën zijn één letter. Dezelfde letter betekent iets anders per kolom: `f` = *flat* bij `cap_shape`, maar *none* bij `gill_color`.

**Keuze:** codes vertalen naar woorden volgens de UCI-documentatie.

**Waarom:** tabellen en grafieken in de EDA worden meteen leesbaar, en de frontend kan dezelfde labels in de dropdowns tonen. Voor het model maakt het geen verschil: een categorie blijft een categorie.

In [20]:
COLORS = {
    "n": "brown", "b": "buff", "g": "gray", "r": "green", "p": "pink", "u": "purple",
    "e": "red", "w": "white", "y": "yellow", "l": "blue", "o": "orange", "k": "black",
}
SURFACES = {
    "i": "fibrous", "g": "grooves", "y": "scaly", "s": "smooth", "h": "shiny", "l": "leathery",
    "k": "silky", "t": "sticky", "w": "wrinkled", "e": "fleshy",
}
NONE = {"f": "none"}

CODE_MAPS = {
    "class": {"e": "edible", "p": "poisonous"},
    "cap_shape": {"b": "bell", "c": "conical", "x": "convex", "f": "flat", "s": "sunken", "p": "spherical", "o": "others"},
    "gill_color": COLORS | NONE,
    "stem_surface": SURFACES | NONE,
    "spore_print_color": COLORS,
    "ring_type": {"c": "cobwebby", "e": "evanescent", "r": "flaring", "g": "grooved", "l": "large", "p": "pendant",
                  "s": "sheathing", "z": "zone", "y": "scaly", "m": "movable", "f": "none", "?": "unknown"},
    "habitat": {"g": "grasses", "l": "leaves", "m": "meadows", "p": "paths", "h": "heaths", "u": "urban", "w": "waste", "d": "woods"},
    "season": {"s": "spring", "u": "summer", "a": "autumn", "w": "winter"},
}

for col, mapping in CODE_MAPS.items():
    mapping = mapping | {"unknown": "unknown"}  # onze eigen categorie uit stap 6d blijft behouden
    unmapped = set(df[col].unique()) - set(mapping)
    assert not unmapped, f"{col}: codes zonder vertaling {unmapped}"  # niets mag stilletjes NaN worden
    df[col] = df[col].map(mapping)

df.head()

,cap_diameter_cm,stem_height_cm,stem_width_mm,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,class
0,NaN,6.42,13.51,unknown,unknown,woods,summer,grooved,convex,unknown,poisonous
1,6.99,7.68,18.76,unknown,white,woods,autumn,none,flat,smooth,edible
2,5.78,7.67,8.21,unknown,green,woods,summer,none,conical,unknown,poisonous
3,6.34,8.34,11.06,unknown,unknown,woods,autumn,zone,convex,unknown,poisonous
4,NaN,5.67,12.78,unknown,brown,woods,unknown,none,convex,unknown,edible


## 9. Datatypes: (geordende) categoricals

**Keuze:** alle categorische kolommen worden `category`-dtype. Dat neemt minder geheugen in, en PyCaret herkent ze automatisch als categorisch.

**Geordend of niet?**
- Kleuren, vormen, oppervlakken en habitats hebben **geen rangorde**.
- `season` heeft wel een volgorde (lente → zomer → herfst → winter), maar die is **cyclisch**: winter ligt naast lente. We maken het een *ordered categorical* zodat tabellen in een logische volgorde staan. In de modellen coderen we het **niet** als 0-1-2-3.
- `class` krijgt de vaste volgorde `edible` < `poisonous`, zodat `poisonous` altijd de positieve klasse (1) is. Recall op giftig is onze belangrijkste metric.

In [21]:
for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("category")

df["season"] = pd.Categorical(df["season"], categories=["spring", "summer", "autumn", "winter", "unknown"], ordered=True)
df["class"] = pd.Categorical(df["class"], categories=["edible", "poisonous"], ordered=True)
df.dtypes

cap_diameter_cm       float64
stem_height_cm        float64
stem_width_mm         float64
spore_print_color    category
gill_color           category
habitat              category
season               category
ring_type            category
cap_shape            category
stem_surface         category
class                category
dtype: object

## 10. Eindcontrole: is de data verbeterd?

In [22]:
pd.DataFrame({
    "ruw": {
        "rijen": len(raw),
        "kolommen": raw.shape[1],
        "ruiskolommen": 2,
        "NaN-cellen (categorisch)": int(raw.drop(columns=["cap-diameter", "stem-height", "stem-width"]).isna().sum().sum()),
        "NaN-cellen (numeriek)": int(raw[["cap-diameter", "stem-height", "stem-width"]].isna().sum().sum()),
        "leesbare labels": "nee",
    },
    "clean": {
        "rijen": len(df),
        "kolommen": df.shape[1],
        "ruiskolommen": 0,
        "NaN-cellen (categorisch)": int(df[CATEGORICAL_COLS].isna().sum().sum()),
        "NaN-cellen (numeriek)": int(df[NUMERIC_COLS].isna().sum().sum()),
        "leesbare labels": "ja",
    },
})

,ruw,clean
rijen,5000,5000
kolommen,13,11
ruiskolommen,2,0
NaN-cellen (categorisch),12860,0
NaN-cellen (numeriek),3440,3426
leesbare labels,nee,ja


Er gingen **geen rijen** verloren. De overblijvende numerieke NaN's worden bewust pas in de modelpipeline ingevuld (stap 6e).

## 11. Opslaan

**Keuze:** we slaan het resultaat op in twee formaten:
- **Parquet**: bewaart de datatypes (categoricals, ordening van `season`/`class`). Dit is het formaat voor de volgende notebooks.
- **CSV**: universeel leesbaar, bv. voor AWS SageMaker of een backend in een andere taal. De datatypes gaan hierbij wel verloren.

We voegen een kolom `id` toe (het rijnummer in het ruwe bestand). Die is **geen feature**, maar maakt de train/test-split hieronder reproduceerbaar.

In [23]:
TARGET = "class"
FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

mushrooms_clean = df.rename_axis("id").reset_index()[["id", TARGET] + FEATURES]

OUT_DIR.mkdir(parents=True, exist_ok=True)
mushrooms_clean.to_parquet(OUT_DIR / "mushrooms_clean.parquet", index=False)
mushrooms_clean.to_csv(OUT_DIR / "mushrooms_clean.csv", index=False)

# Controle: komt de parquet-file identiek terug binnen?
pd.testing.assert_frame_equal(pd.read_parquet(OUT_DIR / "mushrooms_clean.parquet"), mushrooms_clean)
print("Opgeslagen:", mushrooms_clean.shape, "→", OUT_DIR.resolve())

Opgeslagen: (5000, 12) → C:\Users\kobev\Documents\semester 1\3APPAI - Deep Learning\CloudAI\mushrooms\data\clean


## 12. Vaste train/test-split voor alle modellen

**Keuze:** 20% van de rijen (1.000) gaat naar een **vaste testset**, gestratificeerd op de klasse. We slaan de `id`'s op in `test_ids.json`.

**Waarom:**
- Zo gebruiken **alle** modellen (decision tree, random forest, PyCaret, boosting, stacking en later AWS) exact dezelfde testset, en zijn de resultaten eerlijk vergelijkbaar.
- *Stratified*: de verhouding eetbaar/giftig is gelijk in train en test.
- Modelkeuze en tuning gebeuren met **cross-validation op de trainset**. De testset gebruiken we pas helemaal op het einde, één keer.

> **Beperking:** de dataset komt uit 173 soorten, maar er is geen soortkolom. Paddenstoelen van dezelfde soort kunnen dus in train én test zitten.
> De testscore meet daardoor *"hoe goed herkent het model een exemplaar van een soort die al in de trainset zat"*. Voor een volledig onbekende soort zal het model slechter doen.

In [24]:
_, test_ids = train_test_split(
    mushrooms_clean["id"], test_size=0.2, stratify=mushrooms_clean[TARGET], random_state=SEED
)
test_ids = sorted(test_ids.tolist())
(OUT_DIR / "test_ids.json").write_text(json.dumps({"test_ids": test_ids, "seed": SEED, "stratified_on": TARGET}))

is_test = mushrooms_clean["id"].isin(test_ids)
pd.DataFrame({
    "rijen": [(~is_test).sum(), is_test.sum()],
    "% giftig": [round((mushrooms_clean.loc[~is_test, TARGET] == "poisonous").mean() * 100, 1),
                 round((mushrooms_clean.loc[is_test, TARGET] == "poisonous").mean() * 100, 1)],
}, index=["train", "test"])

,rijen,% giftig
train,4000,37.8
test,1000,37.9


## Samenvatting van alle keuzes

| # | Probleem | Keuze | Reden |
|---|---|---|---|
| 2 | Koppeltekens, eenheden verborgen | `snake_case` + eenheid in de naam (`_cm`, `_mm`) | Leesbaar, geen eenheidsfouten in de frontend |
| 3 | `jumbled_noise_0/1` | Verwijderd | Bewezen: rij per rij geschudde kopieën van `cap_shape`, geen echt verband met de klasse |
| 4 | Mogelijk omgedraaide labels | Behouden | Niet op te sporen zonder soortinfo; modellen moeten regulariseren, 100% is onmogelijk |
| 5 | 5 duplicaten (na verwijderen ruis) | Behouden | Verschillende paddenstoelen die door NaN's identiek lijken, geen conflicten |
| 6b | NaN: structureel of willekeurig? | Getoetst met chi² | Enkel `stem_surface` en `spore_print_color` zijn informatief |
| 6c | Ontbrekende steelwaarden bij paddenstoelen zonder steel | Ingevuld met `0` / `none` | Logische afleiding, regel 0× geschonden |
| 6d | NaN in categorische kolommen | Eigen categorie `"unknown"` | Geen kenmerken verzinnen; "niet beschreven" kan informatief zijn |
| 6e | NaN in numerieke kolommen | Laten staan, mediaan in de modelpipeline | Geen leakage van de testset |
| 7 | Grote afmetingen | Behouden, geen clipping | Echte grote soorten; scheve verdeling ≠ fout |
| 8 | Eenletter-codes | Vertaald naar woorden | Leesbaar voor EDA en frontend |
| 9 | Datatypes | `category`; `season` en `class` geordend | `poisonous` = positieve klasse |
| 12 | Train/test | Vaste 80/20-split, gestratificeerd, `test_ids.json` | Alle modellen op dezelfde testset |

**Volgende stap:** `02_modeling.ipynb`.